# 01. Data Loading & Pipeline Integrity

**Project:** Sign Language Hand Gesture Recognition  
**Data Source:** MediaPipe Hand Landmark Keypoints  
**Goal:**  
- Load raw landmark dataset  
- Verify data integrity  
- Perform stratified train–test split  
- Ensure no data leakage  


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [4]:
DATA_PATH = Path("../model_artifacts/keypoint.csv")

df = pd.read_csv(DATA_PATH)

print("✅ Raw dataset loaded successfully")
print("Dataset shape:", df.shape)

df.head()


✅ Raw dataset loaded successfully
Dataset shape: (16413, 43)


,class,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41
0,16,0.0,0.0,-0.101695,-0.127119,-0.127119,-0.338983,-0.144068,-0.500000,-0.177966,...,0.237288,-0.898305,0.203390,-0.288136,0.237288,-0.457627,0.254237,-0.576271,0.271186,-0.677966
1,3,0.0,0.0,-0.240385,-0.019231,-0.442308,0.076923,-0.615385,0.182692,-0.759615,...,-0.125000,0.980769,0.038462,0.480769,0.105769,0.663462,0.163462,0.788462,0.211538,0.884615
2,21,0.0,0.0,-0.295775,-0.126761,-0.633803,-0.084507,-0.816901,0.112676,-0.901408,...,-0.464789,0.267606,-0.225352,0.154930,-0.521127,0.464789,-0.478873,0.464789,-0.380282,0.366197
3,8,0.0,0.0,-0.130631,-0.211712,-0.166667,-0.427928,-0.189189,-0.612613,-0.216216,...,0.220721,-0.364865,0.355856,-0.225225,0.481982,-0.432432,0.540541,-0.540541,0.576577,-0.626126
4,7,0.0,0.0,-0.123288,-0.109589,-0.109589,-0.260274,-0.054795,-0.410959,0.000000,...,0.561644,-0.945205,0.246575,-0.452055,0.356164,-0.630137,0.424658,-0.753425,0.493151,-0.849315


In [5]:
print("📌 Dataset information:")
df.info()


📌 Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16413 entries, 0 to 16412
Data columns (total 43 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   class       16413 non-null  int64  
 1   feature_0   16413 non-null  float64
 2   feature_1   16413 non-null  float64
 3   feature_2   16413 non-null  float64
 4   feature_3   16413 non-null  float64
 5   feature_4   16413 non-null  float64
 6   feature_5   16413 non-null  float64
 7   feature_6   16413 non-null  float64
 8   feature_7   16413 non-null  float64
 9   feature_8   16413 non-null  float64
 10  feature_9   16413 non-null  float64
 11  feature_10  16413 non-null  float64
 12  feature_11  16413 non-null  float64
 13  feature_12  16413 non-null  float64
 14  feature_13  16413 non-null  float64
 15  feature_14  16413 non-null  float64
 16  feature_15  16413 non-null  float64
 17  feature_16  16413 non-null  float64
 18  feature_17  16413 non-null  float64
 19  fe

In [6]:
print("📊 Statistical summary of dataset:")
df.describe().T


📊 Statistical summary of dataset:


,count,mean,std,min,25%,50%,75%,max
class,16413.0,10.666484,6.469300,0.000000,5.000000,11.000000,16.000000,21.000000
feature_0,16413.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
feature_1,16413.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
feature_2,16413.0,-0.019715,0.168528,-0.535714,-0.158824,-0.050000,0.129032,0.521739
feature_3,16413.0,-0.084185,0.140212,-0.464286,-0.180180,-0.109756,-0.010000,0.387500
feature_4,16413.0,-0.029028,0.305629,-0.869565,-0.280899,-0.079365,0.227273,0.869565
feature_5,16413.0,-0.172780,0.284959,-0.785714,-0.361111,-0.251282,-0.020408,0.692308
feature_6,16413.0,-0.032222,0.408048,-1.000000,-0.361446,-0.069444,0.303797,1.000000
feature_7,16413.0,-0.216527,0.415387,-1.000000,-0.491525,-0.337500,0.026316,0.884615
feature_8,16413.0,-0.037178,0.486338,-1.000000,-0.433962,-0.054054,0.358491,1.000000


In [7]:
expected_classes = list(range(22))
actual_classes = sorted(df['class'].unique())

print("Expected classes:", expected_classes)
print("Actual classes  :", actual_classes)

assert actual_classes == expected_classes, "❌ Class label mismatch!"
print("✅ All 22 gesture classes verified")


Expected classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Actual classes  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
✅ All 22 gesture classes verified


In [8]:
print("📈 Class distribution:")
df['class'].value_counts().sort_index()


📈 Class distribution:


class
0     832
1     846
2     561
3     859
4     732
5     814
6     442
7     668
8     890
9     733
10    341
11    863
12    847
13    625
14    797
15    741
16    844
17    873
18    637
19    868
20    887
21    713
Name: count, dtype: int64

In [9]:
X = df.iloc[:, 1:].values   # 42 landmark features
y = df.iloc[:, 0].values   # class labels

print("Feature matrix shape:", X.shape)
print("Label vector shape  :", y.shape)


Feature matrix shape: (16413, 42)
Label vector shape  : (16413,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Stratified split completed")
print("Train samples:", X_train.shape[0])
print("Test samples :", X_test.shape[0])


✅ Stratified split completed
Train samples: 13130
Test samples : 3283


In [11]:
assert set(y_train) == set(y_test), "❌ Leakage detected!"
print("✅ No data leakage — all classes present in train & test")


✅ No data leakage — all classes present in train & test


In [12]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("✅ Feature scaling applied consistently (train → test)")


✅ Feature scaling applied consistently (train → test)


In [14]:
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

train_df = pd.DataFrame(X_train_scaled)
train_df["class"] = y_train
train_df.to_csv(OUTPUT_DIR / "train_raw.csv", index=False)

test_df = pd.DataFrame(X_test_scaled)
test_df["class"] = y_test
test_df.to_csv(OUTPUT_DIR / "test_raw.csv", index=False)

print("✅ Train/Test datasets saved for further processing")


✅ Train/Test datasets saved for further processing


### ✅ Summary

- Raw dataset successfully loaded
- Data types and structure verified
- Stratified train–test split performed
- No data leakage detected
- Scaling applied consistently
- Data saved for cleaning stage

➡️ **Next:** `02_data_cleaning.ipynb`
